#### QUERY `CUSTOMERS_VW`
1. CATALOG NAME: gizmo
2. SCHEMA NAME: bronze
3. VIEW NAME: customers_vw

In [0]:
SELECT DISTINCT * 
FROM gizmo.bronze.customers_vw
WHERE customer_id IS NOT NULL
ORDER BY customer_id ASC;

#### CREATE UDF TO SEPERATE FIRST NAME & LAST NAME FROM CUSTOMERS

In [0]:
CREATE OR REPLACE FUNCTION gizmo.silver.first_name(customer_name STRING)
RETURNS STRING
RETURN split(customer_name, ' ')[0];

CREATE OR REPLACE FUNCTION gizmo.silver.last_name(customer_name STRING)
RETURNS STRING
RETURN split(customer_name, ' ')[size(split(customer_name, ' ')) - 1];

In [0]:
DESCRIBE FUNCTION EXTENDED gizmo.silver.first_name;

In [0]:
DESCRIBE FUNCTION EXTENDED gizmo.silver.last_name;

#### NEED TO LATEST CUSTOMERS BASED ON CREATED_TIMESTAMP
- CREATE TABLE TO STORE THE LATEST CUSTOMERS
1. CATALOG NAME: GIZMO
2. SCHEMA NAME: SILVER
3. TABLE NAME: CUSTOMERS

In [0]:
CREATE OR REPLACE TABLE gizmo.silver.customers
AS
SELECT
customer_id,
  first_name,
  last_name,
  date_of_birth,
  email,
  member_since,
  telephone,
  created_timestamp,
  file_path,
  file_name,
  load_timestamp
FROM
(
  SELECT DISTINCT 
  customer_id,
  gizmo.silver.first_name(customer_name) AS first_name,
  gizmo.silver.last_name(customer_name) AS last_name,
  date_of_birth,
  email,
  member_since,
  telephone,
  created_timestamp,
  file_path,
  file_name,
  load_timestamp,
  rank() over (PARTITION BY customer_id ORDER BY created_timestamp DESC) AS rnk
  FROM gizmo.bronze.customers_vw
  WHERE customer_id IS NOT NULL
  ORDER BY customer_id ASC
) tmp 
WHERE rnk = 1;

#### QUERY AND VALIDATE `GIZMO.SILVER.CUSTOMERS`

In [0]:
SELECT * FROM gizmo.silver.customers ORDER BY 1;

In [0]:
DESCRIBE EXTENDED gizmo.silver.customers;

In [0]:
%python
dbutils.notebook.exit("CUSTOMERS LOADED INTO GIZMO.SILVER.CUSTOMERS")